# Infinigen Visual Positioning Demo

Generates nature scenes (forest, desert) + Soviet urban scene with ISR camera.
Requires Colab GPU runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
import sys, os, subprocess
from pathlib import Path
print("Starting Infinigen VisPos demo...")

## 1. Install Blender 5.1 & Dependencies

In [ ]:
# Install Blender 5.1.1 (~350MB download)
if not Path("/opt/blender/blender").exists():
    print("Downloading Blender 5.1.1...")
    subprocess.run("wget -q https://download.blender.org/release/Blender5.1/blender-5.1.1-linux-x64.tar.xz", shell=True)
    subprocess.run("tar -xf blender-5.1.1-linux-x64.tar.xz", shell=True)
    subprocess.run("mv blender-5.1.1-linux-x64 /opt/blender", shell=True)
    subprocess.run("ln -sf /opt/blender/blender /usr/local/bin/blender", shell=True)

BPY = "/opt/blender/5.1/python/bin/python3.13"
subprocess.run(f"{BPY} -m pip install -q Cython gin-config numpy psutil OpenEXR trimesh shapely scipy scikit-image landlab tqdm opencv-python-headless", shell=True)
print("Setup complete")

## 2. Clone Infinigen & Build Marching Cubes

In [ ]:
if not Path("/content/infinigen").exists():
    subprocess.run("git clone --depth 1 https://github.com/vlordier/infinigen.git /content/infinigen", shell=True)

os.chdir("/content/infinigen")
sys.path.insert(0, "/content/infinigen")

# Build Cython extension
os.chdir("infinigen/terrain/marching_cubes")
subprocess.run("rm -f *.so", shell=True)
subprocess.run([BPY, "-m", "cython", "_marching_cubes_lewiner_cy.pyx", "-3"])
py_inc = subprocess.run("python3-config --includes", shell=True, capture_output=True, text=True).stdout.strip()
np_inc = subprocess.run([BPY, "-c", "import numpy; print(numpy.get_include())"], capture_output=True, text=True).stdout.strip()
cmd = f"gcc -shared -fPIC {py_inc} -I{np_inc} -O2 -o _marching_cubes_lewiner_cy.cpython-313-x86_64-linux-gnu.so _marching_cubes_lewiner_cy.c"
subprocess.run(cmd, shell=True)
subprocess.run("ls -lh *.so", shell=True)
os.chdir("/content/infinigen")
os.environ["INFINIGEN_OCMESHER_CLASS"] = "infinigen.OcMesher.ocmesher.OcMesher"
print("Ready to generate")

## 3. Generate Nature Scenes (Forest, Desert)

In [ ]:
import subprocess as sp

scenes = {
    "forest": "infinigen_examples/configs_nature/scene_types/forest.gin",
    "desert": "infinigen_examples/configs_nature/scene_types/desert.gin",
}

for name, cfg in scenes.items():
    print(f"\nGenerating {name}...")
    cmd = (
        f'blender --background '
        f'--python-expr "import sys; sys.path.insert(0,"/content/infinigen"); '
        f'exec(open("infinigen_examples/generate_nature.py").read())" '
        f'-- --seed 0 --task coarse '
        f'-g {cfg} '
        f'-g infinigen_examples/configs_nature/base_nature.gin '
        f'--output_folder /content/output/{name} 2>&1'
    )
    result = sp.run(cmd, shell=True, capture_output=True, text=True, timeout=900)
    print(result.stdout[-200:] if result.stdout else "No output")
    print(result.stderr[-200:] if result.stderr else "No errors")
    print(f"{name}: {len(list(Path(f'/content/output/{name}').rglob('*')))} files")

## 4. Generate Soviet Urban Scene

In [ ]:
import bpy, random, math, json, numpy as np
from mathutils import Vector, Euler

# Clear everything
for obj in list(bpy.data.objects): bpy.data.objects.remove(obj, do_unlink=True)
for mat in list(bpy.data.materials): bpy.data.materials.remove(mat, do_unlink=True)

scene = bpy.context.scene
scene.render.engine = "CYCLES"
scene.cycles.samples = 64
scene.cycles.use_denoising = True
scene.render.resolution_x = 1280
scene.render.resolution_y = 720

# Sky + ground
world = bpy.data.worlds.new("Sky")
world.use_nodes = True
world.node_tree.nodes["Background"].inputs["Strength"].default_value = 3
scene.world = world

bpy.ops.mesh.primitive_plane_add(size=1200, location=(0,0,0))
mat = bpy.data.materials.new("Ground")
mat.use_nodes = True
mat.node_tree.nodes["Principled BSDF"].inputs["Base Color"].default_value = (0.12,0.12,0.14,1)
bpy.context.active_object.data.materials.append(mat)

# Soviet buildings
from infinigen.assets.urban.regional_styles import get_regional_style
style = get_regional_style("soviet")
colors = style.building_color_palette
random.seed(42)

for i in range(15):
    for j in range(15):
        x = -400 + i * 55 + random.uniform(-8, 8)
        y = -400 + j * 55 + random.uniform(-8, 8)
        h = random.uniform(9, 55)
        bpy.ops.mesh.primitive_cube_add(size=1, location=(x, y, h/2))
        b = bpy.context.active_object
        b.scale = (random.uniform(8, 22), random.uniform(8, 22), h)
        mat = bpy.data.materials.new(f"Mat_{i}_{j}")
        mat.use_nodes = True
        c = random.choice(colors)
        rgb = (int(c[1:3],16)/255, int(c[3:5],16)/255, int(c[5:7],16)/255, 1)
        mat.node_tree.nodes["Principled BSDF"].inputs["Base Color"].default_value = rgb
        b.data.materials.append(mat)

# Mild earthquake
for obj in bpy.data.objects:
    if obj.type == "MESH" and obj.name != "Plane":
        if random.random() < 0.12:
            obj.location.x += random.uniform(-1, 1)
            obj.location.y += random.uniform(-1, 1)
            obj.rotation_euler.z += random.uniform(-0.05, 0.05)

# ISR camera
bpy.ops.object.empty_add(type="PLAIN_AXES", location=(0, 0, 400))
rig = bpy.context.active_object
bpy.ops.object.camera_add()
cam = bpy.context.active_object
cam.parent = rig
cam.rotation_euler = Euler((math.radians(60), 0, 0))
scene.camera = cam

for frame, angle in enumerate(np.linspace(0, 2*np.pi, 4, endpoint=False)):
    rig.location = Vector((250*math.cos(angle), 250*math.sin(angle), 400))
    scene.render.filepath = f"/content/output/urban_{frame:03d}.png"
    bpy.ops.render.render(write_still=True)
    print(f"Frame {frame+1}/4")

# Metadata
meta = {"scene":"soviet_urban","buildings":225,"camera":"ISR orbit 400m","damage":"mild","frames":4}
Path("/content/output/urban_meta.json").write_text(json.dumps(meta, indent=2))
print("Urban scene complete")

## 5. View Results

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Nature — show file counts + first image if available
for name in ["forest", "desert"]:
    imgs = list(Path(f"/content/output/{name}").rglob("*.png"))
    exrs = list(Path(f"/content/output/{name}").rglob("*.exr"))
    print(f"{name}: {len(imgs)} images, {len(exrs)} terrain EXR assets")
    if imgs:
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(imgs[0]))
        plt.title(f"Generated {name} scene")
        plt.axis("off")
        plt.show()

# Urban — show all 4 orbit views
imgs = sorted(Path("/content/output/").glob("urban_*.png"))
if imgs:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, (ax, p) in enumerate(zip(axes, imgs)):
        ax.imshow(Image.open(p))
        ax.set_title(f"Orbit {i*90}")
        ax.axis("off")
    plt.suptitle("Soviet Urban — ISR Camera at 400m", fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Download

In [ ]:
import shutil
shutil.make_archive("/content/infinigen_output", "zip", "/content/output")
from google.colab import files
files.download("/content/infinigen_output.zip")